# DeepGenome Ranking Figures

This notebook renders Fig. 2d-f and the stratified Supplementary Fig. 10-13
panels from privacy-safe frozen aggregates. The private expert-level ranking
TSV is not required for plotting.

## Reproducibility Contract

`PhytoBench-Gene-for_plot/frozen/` is the plotting source of truth. Its
`provenance.json` records source checksums, model columns, row coverage, and
the scoring notebook used to create the aggregate tables. Recreate those
tables with `python -m scripts.freeze_deepgenome_rankings` only when the
private ranking TSV changes.

The legacy Supplementary output prefixes are retained until the authoritative
10-13 panel-to-filename mapping is supplied.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as offline


SAVE_FIGS = os.getenv("PHYTOMNI_SAVE") == "1"
DATA_DIR = Path("PhytoBench-Gene-for_plot/frozen")
OUTPUT_DIR = Path("output")
if SAVE_FIGS:
    OUTPUT_DIR.mkdir(exist_ok=True)

offline.init_notebook_mode(connected=True)
pio.kaleido.scope.default_format = "pdf"

In [ ]:
MODEL_ORDER = ["Phytomni", "Gemini", "Claude", "OpenAI", "Grok"]
MODEL_LABELS = {
    "Phytomni": "Phytomni",
    "Gemini": "Gemini Deep Research",
    "OpenAI": "ChatGPT Agent mode",
    "Grok": "Grok DeepSearch",
    "Claude": "Claude deep research",
}
RANK_ORDER = ["R1", "R2", "R3", "R4", "R5"]
RANK_COLORS = [
    "rgb(0,152,210)",
    "rgb(99,179,228)",
    "rgb(159,204,242)",
    "rgb(207,229,248)",
    "rgb(230,242,250)",
]
FOCAL_COLOR = "rgb(31,113,179)"
CONTEXT_COLOR = "rgb(208,210,211)"
LINE_WIDTH = 2

STRATIFIED_SCOPES = [
    "well_studied",
    "well_studied.rice",
    "well_studied.maize",
    "well_studied.wheat",
    "well_studied.soybean",
    "well_studied.arabidopsis",
    "uncharacterized",
    "uncharacterized.rice",
    "uncharacterized.maize",
    "uncharacterized.wheat",
    "uncharacterized.soybean",
    "uncharacterized.arabidopsis",
    "rice",
    "maize",
    "wheat",
    "soybean",
    "arabidopsis",
]

## Load and Validate Frozen Results

In [ ]:
rank_data = pd.read_csv(DATA_DIR / "rank_distribution.tsv", sep="	")
score_data = pd.read_csv(DATA_DIR / "pl_scores.tsv", sep="	")
pairwise_data = pd.read_csv(DATA_DIR / "pl_pairwise.tsv", sep="	")
provenance = json.loads((DATA_DIR / "provenance.json").read_text())

required_scopes = {"overall", *STRATIFIED_SCOPES}
for name, frame in {
    "rank_distribution": rank_data,
    "pl_scores": score_data,
    "pl_pairwise": pairwise_data,
}.items():
    missing_scopes = required_scopes - set(frame["Scope"])
    if missing_scopes:
        raise ValueError(f"{name} is missing scopes: {sorted(missing_scopes)}")

if set(provenance["model_columns"]) != set(MODEL_ORDER):
    raise ValueError("Frozen model columns do not match the figure configuration.")
if provenance["skipped_rows"] != 0:
    raise ValueError("Frozen results contain skipped ranking rows.")

rank_sums = rank_data.groupby(["Scope", "Model"])["Fraction"].sum()
if not (rank_sums.sub(1.0).abs() < 1e-12).all():
    raise ValueError("Rank fractions must sum to one for every scope and model.")

print(
    f"Frozen source: {provenance['source']['rows']} rows; "
    f"SHA-256 {provenance['source']['sha256']}"
)
print(f"Models: {', '.join(MODEL_ORDER)}")
print(f"Scopes: {len(required_scopes)}")

## Plotting Functions

In [ ]:
def display_labels() -> list[str]:
    return [MODEL_LABELS[model] for model in MODEL_ORDER]


def save_figure(figure: go.Figure, file_prefix: str) -> None:
    if not SAVE_FIGS:
        return
    figure.write_image(OUTPUT_DIR / f"{file_prefix}.pdf")
    figure.write_image(OUTPUT_DIR / f"{file_prefix}.png")


def rank_distribution_figure(scope: str) -> go.Figure:
    scope_data = rank_data[rank_data["Scope"] == scope]
    rank_matrix = (
        scope_data.pivot(index="Model", columns="Rank", values="Fraction")
        .reindex(index=MODEL_ORDER, columns=RANK_ORDER)
    )
    if rank_matrix.isna().any().any():
        raise ValueError(f"Incomplete rank distribution for scope {scope!r}.")

    figure = go.Figure()
    for rank_label, color in zip(RANK_ORDER, RANK_COLORS, strict=True):
        figure.add_trace(
            go.Bar(
                x=display_labels(),
                y=100.0 * rank_matrix[rank_label],
                name=rank_label.replace("R", "Rank "),
                marker_color=color,
            )
        )
    figure.update_layout(
        barmode="stack",
        showlegend=False,
        xaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
        },
        yaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
            "title_text": "Percent (%)",
            "range": [0, 100],
        },
        plot_bgcolor="white",
        font_family="Arial",
        font_color="rgb(0,0,0)",
        font_size=20,
        width=1080,
        height=1080,
    )
    return figure


def pairwise_probability_figure(scope: str) -> go.Figure:
    scope_data = pairwise_data[pairwise_data["Scope"] == scope]
    probability_matrix = (
        scope_data.pivot(
            index="RowModel",
            columns="ColumnModel",
            values="Probability",
        )
        .reindex(index=MODEL_ORDER, columns=MODEL_ORDER)
    )
    if probability_matrix.shape != (len(MODEL_ORDER), len(MODEL_ORDER)):
        raise ValueError(f"Incomplete pairwise matrix for scope {scope!r}.")

    figure = go.Figure(
        go.Heatmap(
            x=display_labels(),
            y=display_labels(),
            z=probability_matrix.to_numpy(),
            colorscale="TroPic",
            zmin=0,
            zmax=1,
            zmid=0.5,
        )
    )
    figure.update_layout(
        showlegend=False,
        yaxis_scaleanchor="x",
        xaxis={"showline": False, "ticks": "outside"},
        yaxis={"showline": False, "ticks": "outside"},
        plot_bgcolor="white",
        font_family="Arial",
        font_color="rgb(0,0,0)",
        font_size=20,
        width=1080,
        height=1080,
    )
    return figure


def elo_score_figure(scope: str) -> go.Figure:
    scope_scores = (
        score_data[score_data["Scope"] == scope]
        .set_index("Model")
        .reindex(MODEL_ORDER)
    )
    if scope_scores["Elo"].isna().any():
        raise ValueError(f"Incomplete Elo scores for scope {scope!r}.")

    values = scope_scores["Elo"].to_numpy()
    colors = [FOCAL_COLOR, *([CONTEXT_COLOR] * (len(MODEL_ORDER) - 1))]
    figure = go.Figure(
        go.Bar(
            x=display_labels(),
            y=values,
            marker_color=colors,
            marker_line_color="rgb(0,0,0)",
            marker_line_width=LINE_WIDTH,
            text=[f"{value:.0f}" for value in values],
            textposition="outside",
        )
    )
    figure.update_layout(
        showlegend=False,
        xaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
        },
        yaxis={
            "showline": True,
            "linewidth": LINE_WIDTH,
            "linecolor": "rgb(0,0,0)",
            "mirror": True,
            "ticks": "outside",
            "tickwidth": LINE_WIDTH,
            "title_text": "Score",
            "range": [1000, 2000],
        },
        plot_bgcolor="white",
        font_family="Arial",
        font_color="rgb(0,0,0)",
        font_size=20,
        width=1080,
        height=1080,
    )
    return figure

## Fig. 2d-f

In [ ]:
fig_2d = rank_distribution_figure("overall")
fig_2d.show()
save_figure(fig_2d, "fig.2d.phytobench-gene.percent.bar")

fig_2e = pairwise_probability_figure("overall")
fig_2e.show()
save_figure(fig_2e, "fig.2e.phytobench-gene.prob.heatmap")

fig_2f = elo_score_figure("overall")
fig_2f.show()
save_figure(fig_2f, "fig.2f.phytobench-gene.score.bar")

## Supplementary Fig. 10-13 Stratified Panels

Each scope emits a rank-distribution bar chart, a pairwise-probability heatmap,
and an Elo-like score bar chart.

In [ ]:
for scope in STRATIFIED_SCOPES:
    rank_figure = rank_distribution_figure(scope)
    rank_figure.show()
    save_figure(
        rank_figure,
        f"supplementary_fig.7.phytobench-gene.{scope}.percent.bar",
    )

    probability_figure = pairwise_probability_figure(scope)
    probability_figure.show()
    save_figure(
        probability_figure,
        f"supplementary_fig.8.phytobench-gene.{scope}.prob.heatmap",
    )

    score_figure = elo_score_figure(scope)
    score_figure.show()
    save_figure(
        score_figure,
        f"supplementary_fig.9.phytobench-gene.{scope}.score.bar",
    )